# Disease relevance and mechanism classes of the translated lincRNA ORFs

Continues [Liver_lincRNA_ORFs.ipynb](Liver_lincRNA_ORFs.ipynb) from the lincorfz run `v2`: every ORF of the run is
annotated with quality flags, disease-relevance measures D1-D6 (summarized as an HCC score and a chronic liver
disease score) and mechanism classes M1-M6 (encoded peptide, translation-coupled RNA regulation, cis regulation,
nuclear trans function, cytoplasmic trans function, small-RNA host). The definitions are in
[lincdisease/LOGIC.md](lincdisease/LOGIC.md), the Methods in `lincdisease_Methods.docx` of the output directory.

* **Heavy computations** are done by the Snakemake workflow [lincdisease/workflow](lincdisease/workflow/README.md),
  driven by `lincdisease/run.sh`: downloads on the login node, everything else as Slurm jobs.
* **This notebook** prints the commands, reads the result tables and plots them. It can itself be executed as a
  Slurm job (last section).
* **Paths** come from `lincdisease/config/config.yaml`, relative to the root directory of the workflow
  (environment variable `LINCDISEASE_ROOT`; default: two levels above this notebook).

In [ ]:
import os
import re
import sys
import gzip
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp())   # the home directory may be read-only
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

LDIR = Path(os.environ.get("LINCDISEASE_DIR", "lincdisease")).resolve()      # lincdisease/ of the repository
ROOT = Path(os.environ.get("LINCDISEASE_ROOT", "../..")).resolve()
sys.path.insert(0, str(LDIR))
from lincdisease import chronic                                               # noqa: E402

cfg = yaml.safe_load(open(LDIR / "config" / "config.yaml"))
at_root = lambda p: Path(p) if os.path.isabs(p) else ROOT / p
OUT = at_root(cfg["out_dir"])
DATA = at_root(cfg["data_dir"])
GOV = at_root(cfg["govaere"]["dir"])
PY = at_root(cfg["python"])
RUN_SH = LDIR / "run.sh"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = {"M1": "encoded peptide", "M2": "translation-coupled RNA", "M3": "cis regulation",
               "M4": "nuclear trans", "M5": "cytoplasmic trans", "M6": "small-RNA host"}
print("root:", ROOT, "\noutputs:", OUT)

## 1. Conda environments and inputs

Compute nodes have no internet access, so the environments are built once on a login node from
`lincdisease/envs/` (conda-forge and bioconda channels directly):

* `lincdisease`: the package, its tests and this notebook (`ipykernel`, `nbclient`, `nbconvert`)
* `lincdisease_peptide`: MHCflurry and the sequence-feature tools of the `peptide` step
* `snakemake`: Snakemake 9 with the cluster-generic executor

The workflow starts from the finished lincorfz run, linked into `<root>/inputs/` (see the workflow README).

In [ ]:
for env in ["lincdisease", "lincdisease_peptide"]:
    print(f"mamba env create -p {ROOT}/conda_envs/{env} -f {LDIR}/envs/{env}.yml")
print(f"mamba create -p {ROOT}/conda_envs/snakemake -c conda-forge -c bioconda python=3.12 snakemake-minimal=9.27.1 "
      "snakemake-executor-plugin-cluster-generic")

inputs = {k: at_root(cfg[k]) for k in ["results_dir", "candidates", "peptides_dir", "phylop"]}
display(pd.DataFrame({"path": {k: str(p) for k, p in inputs.items()},
                      "exists": {k: p.exists() for k, p in inputs.items()}}))

## 2. Run the workflow

From the root directory on a login node. `start` runs in the background (the downloads need the network; all
other steps are Slurm jobs) and resumes where an earlier run stopped; a step reruns when its inputs or the
lincdisease modules it uses change.

In [ ]:
print(f"cd {ROOT}")
for cmd in ["plan", "start", "status"]:
    print(f"{RUN_SH} {cmd}")
print(f"{RUN_SH} redo mechanism     # rerun a step and everything downstream of it")

## 3. Results

### 3.1 Quality layer: confidence tiers and copy attribution

Every ORF gets flags (segmental duplication, GRCh38 false duplication, homology to annotated proteins, antisense
position, small-RNA overlap, multimapper dominance, coding-gene extension), a copy-attribution confidence for ORFs
with paralogous copies, and a tier: clean / caution / expressed_paralog / likely_artefact.

In [ ]:
qc = pd.read_csv(OUT / "qc.orfs.tsv", sep="\t")
tiers = ["clean", "caution", "expressed_paralog", "likely_artefact"]
t = pd.crosstab(qc["confidence_tier"], qc["is_candidate"].map({True: "candidates", False: "other ORFs"}))
display(t.reindex(tiers).fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(5, 3.2))
for tier, color in zip(tiers, ["#2a7f62", "#d9a441", "#c0504d", "#7f7f7f"]):
    v = qc.loc[qc["confidence_tier"] == tier, "copy_attribution"].dropna()
    ax.hist(v, bins=np.linspace(0, 1, 21), histtype="step", lw=1.5, color=color, label=f"{tier} ({len(v)})")
ax.set(xlabel="copy-attribution confidence", ylabel="ORFs", yscale="log")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "qc_copy_attribution.pdf", bbox_inches="tight")

### 3.2 Disease relevance: D1-D6, HCC and chronic liver disease scores

D1 differential expression (HCC: TCGA-LIHC tumour vs normal and vs GTEx liver; chronic: fibrosis stage in MASLD
and HCV cohorts), D2 survival, D3 somatic copy number / methylation, D4 liver-trait GWAS and eQTLs, D5 literature,
D6 liver specificity. Components are percentiles among all quantified loci (0.5 = typical lincRNA).

In [ ]:
orfs = pd.read_csv(OUT / "lincdisease.orfs.tsv", sep="\t", low_memory=False)
cand = pd.read_csv(OUT / "lincdisease.candidates.tsv", sep="\t")
comps = [f"comp_D{i}" for i in range(1, 7)]
c = cand.sort_values("hcc_score", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [3, 1.3]})
im = axes[0].imshow(c[comps].to_numpy(float), aspect="auto", cmap="RdBu_r", vmin=0, vmax=1)
axes[0].set_xticks(range(6), [s.replace("comp_", "") for s in comps])
axes[0].set_yticks(range(len(c)), c["orf_label"], fontsize=7)
fig.colorbar(im, ax=axes[0], shrink=0.4, label="component (percentile)")
y = np.arange(len(c))
axes[1].barh(y - 0.2, c["hcc_score"], height=0.4, color="#c0504d", label="HCC")
axes[1].barh(y + 0.2, c["chronic_score"], height=0.4, color="#4f81bd", label="chronic")
axes[1].axvline(0.5, color="grey", ls=":")
axes[1].set(yticks=[], ylim=axes[0].get_ylim(), xlabel="score", xlim=(0, 1))
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "candidates_disease_components.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
loci = orfs.drop_duplicates("locus_id")
ax.scatter(loci["hcc_score"], loci["chronic_score"], s=3, color="lightgrey", label="all loci")
cl = cand.drop_duplicates("locus_id")
ax.scatter(cl["hcc_score"], cl["chronic_score"], s=18, color="#c0504d", label="candidate loci")
for _, r in cl.nlargest(8, "hcc_score").iterrows():
    ax.annotate(r["locus_name"], (r["hcc_score"], r["chronic_score"]), fontsize=7)
for _, r in cl.nlargest(5, "chronic_score").iterrows():
    ax.annotate(r["locus_name"], (r["hcc_score"], r["chronic_score"]), fontsize=7)
ax.set(xlabel="HCC score", ylabel="chronic liver disease score")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "hcc_vs_chronic_score.pdf", bbox_inches="tight")
cols = ["orf_label", "confidence_tier", "hcc_score", "chronic_score", "log2FC_paired", "fdr_paired",
        "stat_D1_chronic", "chronic_fdr", "HR_OS", "fdr_OS", "gwas_top_trait", "n_liver"]
display(cand[cols].sort_values("hcc_score", ascending=False).round(3))

### 3.3 Positive and negative controls

One or two known genes per measure and score, with the expectation fixed before scoring (GSTP1 was added post hoc
after MEG3 failed as the methylation control). Mechanism controls that are not lincorfz loci (MTLN, SPAAR) cannot
be scored.

In [ ]:
ctrl = pd.read_csv(OUT / "controls.tsv", sep="\t")
status = ctrl["pass"].map({True: "pass", False: "FAIL"}).fillna("not scored")
display(ctrl.assign(result=status).drop(columns="pass").style.apply(
    lambda r: ["background-color: #f4cccc" if r["result"] == "FAIL" else ""] * len(r), axis=1))
print(status.value_counts().to_dict())

### 3.4 Mechanism classes

Each class score is the mean of its feature percentiles (missing features shrunk to 0.5); scores are standardized
over all ORFs (`M*_z`), and the dominant class is the class with the highest z if z > 0 and it exceeds the
runner-up by 0.5 SD, otherwise `none`. M2 uses the NMD substrate rule and the measured NMD sensitivity (3.5).

In [ ]:
mech = pd.read_csv(OUT / "mechanism.orfs.tsv", sep="\t", low_memory=False)
classes = list(CLASS_NAMES)
d = pd.concat({"all ORFs": mech["dominant_class"].value_counts(),
               "candidates": mech.loc[mech["is_candidate"], "dominant_class"].value_counts()}, axis=1)
display(d.reindex(classes + ["none"]).fillna(0).astype(int))

mc = mech.loc[mech["is_candidate"]].set_index("orf_label")
mc = mc.sort_values(["dominant_class"] + [f"{k}_z" for k in classes])
z = mc[[f"{k}_z" for k in classes]].to_numpy(float)
fig, ax = plt.subplots(figsize=(7, 10))
im = ax.imshow(z, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(6), [f"{k}\n{CLASS_NAMES[k]}" for k in classes], fontsize=7)
ax.set_yticks(range(len(mc)), [f"{o}  [{d}]" for o, d in zip(mc.index, mc["dominant_class"])], fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.4, label="standardized class score (z)")
fig.savefig(FIG / "candidates_mechanism_z.pdf", bbox_inches="tight")

### 3.5 Measured NMD sensitivity (M2)

HeLa UPF1 knock-down and SMG6 + SMG7 double knock-down vs scrambled controls (Colombo et al. 2017, recount3
SRP083135); per locus the mean log2 fold change of the two knock-downs. Loci whose ORF places the stop codon
> 50 nt upstream of the last exon junction (predicted NMD substrates) should rise upon NMD inhibition.

In [ ]:
nmd = pd.read_csv(OUT / "nmd.loci.tsv", sep="\t").set_index("feature_id")
pred = mech.groupby("locus_id")["raw_m2_nmd"].max()
v = nmd.join(pred.rename("predicted_nmd"), how="inner").dropna(subset=["nmd_log2fc", "predicted_nmd"])
from scipy import stats
a, b = v.loc[v["predicted_nmd"] == 1, "nmd_log2fc"], v.loc[v["predicted_nmd"] == 0, "nmd_log2fc"]
print(f"predicted NMD substrates: n={len(a)}, median log2FC {a.median():.2f}; others: n={len(b)}, "
      f"median {b.median():.2f}; Mann-Whitney p = {stats.mannwhitneyu(a, b).pvalue:.1e}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
bins = np.linspace(-4, 4, 41)
axes[0].hist(b.clip(-4, 4), bins=bins, density=True, histtype="step", lw=1.5, color="grey", label="other loci")
axes[0].hist(a.clip(-4, 4), bins=bins, density=True, histtype="step", lw=1.5, color="#c0504d",
             label="predicted NMD substrate")
axes[0].set(xlabel="log2FC upon NMD inhibition (mean of UPF1 KD, SMG6+7 dKD)", ylabel="density")
axes[0].legend(frameon=False, fontsize=8)
axes[1].scatter(nmd["log2fc_UPF1_kd"], nmd["log2fc_SMG6_SMG7_kd"], s=3, color="lightgrey")
for fid, r in nmd.loc[nmd.index.str.startswith("ctrl:")].iterrows():
    axes[1].scatter(r["log2fc_UPF1_kd"], r["log2fc_SMG6_SMG7_kd"], s=12, color="k")
    axes[1].annotate(fid[5:], (r["log2fc_UPF1_kd"], r["log2fc_SMG6_SMG7_kd"]), fontsize=7)
axes[1].set(xlabel="log2FC UPF1 KD", ylabel="log2FC SMG6 + SMG7 dKD", title="control genes labelled")
fig.tight_layout()
fig.savefig(FIG / "nmd_sensitivity_validation.pdf", bbox_inches="tight")

### 3.6 ORFs supported by published CRISPR screens

ORFs with a screen value >= 1: viability phenotype of an ORF-targeting guide in Prensner et al. 2021 (2 if the
phenotype is ORF-specific) or a hit of Chen et al. 2020 (re-called against the non-targeting null).

In [ ]:
scr = pd.read_csv(OUT / "lincdisease.screen_supported.tsv", sep="\t")
print(f"{len(scr)} ORFs, {scr['is_candidate'].sum()} of them candidates")
cols = ["orf_label", "is_candidate", "confidence_tier", "crispr_screen_value", "prensner_hepg2_lfc",
        "chen_K562_phenotype", "chen_iPSC_phenotype", "phylop_orf_minus_nonorf", "hcc_score", "chronic_score",
        "dominant_class"]
display(scr[cols].sort_values(["crispr_screen_value", "hcc_score"], ascending=False).round(3))

### 3.7 Govaere et al. 2020 NAFLD cohort (second fibrosis cohort of D1 chronic)

216 liver biopsies (GSE135251, PRJNA558102; NAFL and NASH with fibrosis stage F0-F4, plus controls), aligned with
STAR to GRCh38 / GENCODE v42 and archived as CRAM 3.1 (`<govaere dir>/cram/`), in batches because of storage;
215 are used (the read 2 file of one NASH F2 run is corrupt at ENA). The loci are quantified in the CRAMs over the same regions as TCGA/GTEx and the recount3 cohorts
(`workflow/scripts/govaere_locus_cov.sh`, Slurm), scaled by median-of-ratios size factors of the protein-coding
STAR gene counts (reverse-stranded library) and tested against fibrosis stage and NAFLD activity score (NAS) with
the partial rank correlation of D1 chronic (technical covariates: log input reads, % uniquely mapped reads,
fraction of reads assigned to genes). The cohort enters D1 chronic next to Hoang et al. 2019 (same design:
fibrosis stage and NAS; the fibrosis weight is split between the two by the samples tested); below, the agreement
of the two cohorts.

In [ ]:
# commands (login node): sample annotation from GEO, and locus coverage of the aligned CRAMs
print(f"curl -sS -f -o {GOV}/metadata/GSE135251_samples.soft.txt "
      "'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE135251&targ=gsm&form=text&view=brief'")
print(f"sbatch --output={GOV}/logs/locus_cov.%j.log {LDIR}/workflow/scripts/govaere_locus_cov.sh {GOV} "
      f"{OUT}/quant/regions.tsv {at_root(cfg['govaere']['reference'])}/GRCh38.primary_assembly.genome.fa "
      f"'{cfg['govaere']['modules']}'")

# GEO sample characteristics -> runs
rec, cur = [], None
for line in open(GOV / "metadata" / "GSE135251_samples.soft.txt"):
    if line.startswith("^SAMPLE"):
        cur = {"gsm": line.split("=")[1].strip()}
        rec.append(cur)
    elif line.startswith("!Sample_characteristics_ch1") and cur is not None:
        k, _, val = line.split("=", 1)[1].strip().partition(": ")
        cur[k] = val
meta = pd.DataFrame(rec)
runs = pd.read_csv(LDIR / cfg["govaere"]["runs"], sep="\t")
gov = runs[["run_accession", "experiment_alias"]].merge(meta, left_on="experiment_alias", right_on="gsm")
gov = gov.set_index("run_accession")
gov["fibrosis_stage"] = pd.to_numeric(gov["fibrosis stage"], errors="coerce")
gov["nas"] = pd.to_numeric(gov["nas score"], errors="coerce")
gov["aligned"] = [(GOV / "cram" / f"{r}.ok").exists() for r in gov.index]
gov["quantified"] = [(GOV / "locus_cov" / f"{r}.tsv").exists() for r in gov.index]
display(gov.groupby("group in paper")[["aligned", "quantified"]].agg(["sum", "size"]))

In [ ]:
# STAR summary of the aligned runs
def star_log(run):
    d = {}
    for line in open(GOV / "star_logs" / f"{run}.Log.final.out"):
        if "|" in line:
            k, v = (s.strip() for s in line.split("|", 1))
            d[k] = v
    return {"input_reads": int(d["Number of input reads"]), "read_len": int(d["Average input read length"]),
            "pct_unique": float(d["Uniquely mapped reads %"].rstrip("%")),
            "pct_multi": float(d["% of reads mapped to multiple loci"].rstrip("%"))}

done = gov.index[gov["quantified"]]
star = pd.DataFrame({r: star_log(r) for r in done}).T
counts = pd.concat({r: pd.read_csv(GOV / "counts" / f"{r}.ReadsPerGene.tsv", sep="\t", header=None, index_col=0)[3]
                    for r in done}, axis=1)            # column 4: reverse-stranded library
counts = counts.loc[~counts.index.str.startswith("N_")]
star["frac_assigned"] = counts.sum() / star["input_reads"]
display(star.describe().round(2))

In [ ]:
# protein-coding genes (size factors) and gene symbols (cell-type markers) from the GENCODE v42 annotation
genes = []
with gzip.open(DATA / "gencode.v42.annotation.gtf.gz", "rt") as fh:
    for line in fh:
        if "\tgene\t" in line:
            a = line.split("\t")[8]
            genes.append((re.search('gene_id "([^"]+)"', a).group(1), re.search('gene_type "([^"]+)"', a).group(1),
                          re.search('gene_name "([^"]+)"', a).group(1)))
genes = pd.DataFrame(genes, columns=["gene_id", "gene_type", "gene_name"]).set_index("gene_id")
pc = counts.loc[counts.index.intersection(genes.index[genes["gene_type"] == "protein_coding"])]
sf = chronic.size_factors(pc)

ncov = pd.concat({r: pd.read_csv(GOV / "locus_cov" / f"{r}.tsv", sep="\t", index_col=0)["ncov"] for r in done},
                 axis=1).fillna(0)
# ncov is summed per-base depth: divided by the read-pair length it is ~ fragments
expr = np.log2(ncov.div(sf * star["read_len"], axis=1) + 0.05)
tech = pd.DataFrame({"log_input_reads": np.log10(star["input_reads"]), "pct_unique": star["pct_unique"],
                     "frac_assigned": star["frac_assigned"]})
detected = (ncov > 0).mean(axis=1) >= chronic.MIN_DETECTED
res = {}
for y in ["fibrosis_stage", "nas"]:
    t = chronic.partial_rank(expr.loc[detected], gov.loc[done, y].astype(float), tech)
    res[f"govaere_{y}_r"], res[f"govaere_{y}_p"] = t["stat"], t["p"]
govres = pd.DataFrame(res)
govres["govaere_n"] = gov.loc[done, "fibrosis_stage"].notna().sum()
print(f"{len(done)} runs quantified; {detected.sum()} of {len(ncov)} features detected in >= "
      f"{chronic.MIN_DETECTED:.0%} of them")
display(govres.loc[govres.index.str.startswith("ctrl:")].round(3).sort_values("govaere_fibrosis_stage_r"))

In [ ]:
# replication of the Hoang et al. 2019 fibrosis association (D1 chronic) and the candidates
d1c = pd.read_csv(OUT / "D1.chronic.tsv", sep="\t").set_index("feature_id")
both = govres.join(d1c[["hoang_r", "hoang_p", "D1_chronic", "chronic_fdr"]], how="inner").dropna(
    subset=["govaere_fibrosis_stage_r", "hoang_r"])
rho = stats.spearmanr(both["govaere_fibrosis_stage_r"], both["hoang_r"])
print(f"Govaere vs Hoang fibrosis statistics over {len(both)} features: Spearman {rho.statistic:.2f} "
      f"(p = {rho.pvalue:.1e})")
if "ctrl:COL1A1" in both.index:
    print("COL1A1:", both.loc["ctrl:COL1A1", ["govaere_fibrosis_stage_r", "govaere_fibrosis_stage_p",
                                              "hoang_r", "hoang_p"]].round(4).to_dict())
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(both["hoang_r"], both["govaere_fibrosis_stage_r"], s=4, color="lightgrey")
cl_ids = set(cand["locus_id"])
cb = both.loc[both.index.isin(cl_ids)]
ax.scatter(cb["hoang_r"], cb["govaere_fibrosis_stage_r"], s=18, color="#c0504d", label="candidate loci")
if "ctrl:COL1A1" in both.index:
    r = both.loc["ctrl:COL1A1"]
    ax.scatter(r["hoang_r"], r["govaere_fibrosis_stage_r"], s=30, color="k", marker="^", label="COL1A1")
lim = (-0.8, 0.8)
ax.plot(lim, lim, ls=":", color="grey")
ax.set(xlim=lim, ylim=lim, xlabel="Hoang 2019: partial r with fibrosis stage",
       ylabel=f"Govaere 2020 ({len(done)} runs): partial r with fibrosis stage",
       title=f"{len(both)} features, Spearman {rho.statistic:.2f} (p = {rho.pvalue:.1e})")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "govaere_vs_hoang_fibrosis.pdf", bbox_inches="tight")

cg = cand.drop_duplicates("locus_id").set_index("locus_id")[["locus_name", "chronic_score"]].join(both, how="left")
display(cg.sort_values("govaere_fibrosis_stage_p").round(3))

### 3.8 Final table

`lincdisease.table.html` (output directory) holds the candidates and the screen-supported ORFs with all scores,
sortable, with column groups that can be hidden and an index of the scores; the tab-separated tables
`lincdisease.candidates.tsv`, `lincdisease.screen_supported.tsv` and `lincdisease.orfs.tsv` (all ORFs) have the
same columns.

In [ ]:
print(OUT / "lincdisease.table.html")
cols = ["orf_label", "confidence_tier", "n_aa", "ribo_evidence", "hcc_score", "chronic_score", "actionability_score",
        "dominant_class"] + [f"{k}_z" for k in CLASS_NAMES]
display(cand[cols].sort_values("hcc_score", ascending=False).round(2))

## 4. Execute this notebook as a Slurm job

A copy is executed in the output directory, so that the notebook in the repository stays without outputs; the
executed copy is exported to HTML next to it.

In [ ]:
nb_dir = OUT / "notebook"
logs = nb_dir / "slurm_logs"
logs.mkdir(parents=True, exist_ok=True)
nb = "Liver_lincRNA_ORFs_disease.ipynb"
env = PY.parent.parent
repo = LDIR.parent                                      # Liver_lincRNA_ORFs/ of the repository
print(f"cp {repo / nb} {nb_dir}/ && \\\n"
      f"sbatch --time=01:00:00 --mem=16G --export=ALL,LINCDISEASE_ROOT={ROOT},LINCDISEASE_DIR={LDIR} --output={logs}/notebook_%j.out "
      f"--error={logs}/notebook_%j.err --wrap='bash {repo}/sbatch/execute_notebook.sbatch {nb_dir / nb} {env} && "
      f"{env}/bin/jupyter nbconvert --to html {nb_dir / nb}'")